# Full Combo + Transformer (long training budget) on Kaggle GPU

Clones `approach/full-combo-transformer`, forked from `approach/full-
combo-conv1d` with one swap: the neural signal is now the char-level
Transformer encoder (`approach/transformer`) instead of the Conv1D +
BiLSTM, trained for a much longer budget than the earlier 40-epoch
standalone run (which only reached ~46%). Candidate-filtering, n-gram
fallback, and the vowel-ratio guard are all unchanged from full-combo-
conv1d -- this isolates exactly one variable: does more training time on
a no-recurrence architecture beat the recurrent one at the same blend.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/full-combo-transformer"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train the Transformer

Watch the printed loss per epoch. This is a much longer run than the
40-epoch standalone transformer check (which validated at ~46%) --
epochs is set well above that here since the budget allows it. Watch the
per-epoch `time=` print after the first few epochs to sanity-check the
total runtime before committing to the full run; lower `--epochs` below
if it's tracking longer than you have time for, since the warmup+cosine
schedule automatically rescales to whatever value you pass.

In [ ]:
!python src/train_transformer.py --epochs 150

## Validate the combined agent

Same held-out-train.txt methodology as every other branch. Compare
directly against `full-combo-conv1d`'s 57.6% (100 epochs) -- this is the
number that decides which branch becomes the final submission. Add
`--full` once you want the definitive number on all held-out words
instead of the quick 3000-word sample.

In [ ]:
!python src/validate_combined.py

## Generate submission.csv

Only run this once the validation number is in and you've decided this
branch should be the final submission -- it takes a while (~250,000
words, one game at a time) and there's no reason to spend that time on a
candidate that didn't beat full-combo-conv1d.

In [ ]:
!python src/generate_submission_combined.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/transformer_masker.pt", "/kaggle/working/transformer_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved transformer_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")